In [12]:
import torch
import torch.nn as nn
from torchvision import models, transforms
import cv2
from ultralytics import YOLO

In [10]:

path = "resnet18_fruits.pth"

checkpoint = torch.load(path, map_location="cpu")

print(type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nKeys:")
    print(checkpoint.keys())

<class 'collections.OrderedDict'>

Keys:
odict_keys(['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.bn1.num_batches_tracked', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias', 'layer1.0.bn2.running_mean', 'layer1.0.bn2.running_var', 'layer1.0.bn2.num_batches_tracked', 'layer1.1.conv1.weight', 'layer1.1.bn1.weight', 'layer1.1.bn1.bias', 'layer1.1.bn1.running_mean', 'layer1.1.bn1.running_var', 'layer1.1.bn1.num_batches_tracked', 'layer1.1.conv2.weight', 'layer1.1.bn2.weight', 'layer1.1.bn2.bias', 'layer1.1.bn2.running_mean', 'layer1.1.bn2.running_var', 'layer1.1.bn2.num_batches_tracked', 'layer2.0.conv1.weight', 'layer2.0.bn1.weight', 'layer2.0.bn1.bias', 'layer2.0.bn1.running_mean', 'layer2.0.bn1.running_var', 'layer2.0.bn1.num_batches_tracked', 'layer2.0.conv2.weight', 'lay

In [11]:


# =========================
# SETTINGS
# =========================

MODEL_PATH = "resnet18_fruits.pth"

# Change these to your actual classes
CLASSES = [
    "fresh_apple",
    "fresh_banana",
    "fresh_strewberry",
    "rotten_apple",
    "rotten_banana",
    "rotten_strewberry"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)



model = models.resnet18(weights=None)

model.fc = nn.Linear(
    model.fc.in_features,
    len(CLASSES)
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model.load_state_dict(checkpoint)

model.to(device)
model.eval()

print("Model loaded successfully!")


# =========================
# PREPROCESSING
# =========================

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])


# =========================
# WEBCAM
# =========================

cap = cv2.VideoCapture(1)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")


# =========================
# REAL-TIME LOOP
# =========================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    # BGR → RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Preprocess
    image = transform(rgb)

    # Add batch dimension
    image = image.unsqueeze(0).to(device)

    # Prediction
    with torch.no_grad():

        output = model(image)

        probabilities = torch.softmax(output, dim=1)

        confidence, predicted = torch.max(
            probabilities,
            dim=1
        )

    # Get result
    class_name = CLASSES[predicted.item()]
    confidence = confidence.item() * 100

    # =========================
    # DISPLAY
    # =========================

    text = f"{class_name}: {confidence:.1f}%"

    cv2.putText(
        frame,
        text,
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 255, 0),
        3
    )

    cv2.imshow(
        "Real-Time Fruit Quality Classification",
        frame
    )

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# =========================
# CLEANUP
# =========================

cap.release()
cv2.destroyAllWindows()

Device: cpu
Model loaded successfully!


In [ ]:


# Load your trained YOLO model
model = YOLO("best.pt")

# Open webcam
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam")

while True:

    # Read frame
    ret, frame = cap.read()

    if not ret:
        break

    # YOLO inference
    results = model(frame)

    # Draw detections on frame
    annotated_frame = results[0].plot()

    # Show
    cv2.imshow("YOLO Real-Time Detection", annotated_frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()